# 3 BF Fixed Random-Vector cGAN

Self-contained Colab notebook for `2A_BF_cGANRandomVecFixed` on the CONUS MSA-sampled training archive. Outputs are written to `/content/drive/MyDrive/IM3/EvalP1/revision_msa_sample/2A_BF_cGANRandomVecFixed_MSASample` with fixed run folders.


In [1]:
import torch
print("3 BF Fixed Random-Vector cGAN")
print("Using torch", torch.__version__)
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("No GPU detected. In Colab, use Runtime > Change runtime type > GPU.")

!nvidia-smi


3 BF Fixed Random-Vector cGAN
Using torch 2.11.0+cu128
GPU: NVIDIA A100-SXM4-80GB
Fri Jul  3 15:27:19 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100-SXM4-80GB          Off |   00000000:00:05.0 Off |                    0 |
| N/A   37C    P0             56W /  400W |       6MiB /  81920MiB |      0%      Default |
|                                         |               

In [2]:
import csv
import json
import random
import shutil
import subprocess
import sys
import time
import zipfile
from dataclasses import asdict, dataclass
from pathlib import Path
from typing import Iterable, Optional

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset


In [3]:
# =============================================================================
# Colab configuration. Edit only paths/hyperparameters here if needed.
# These copies target the CONUS MSA-sampled training archive built for the
# revision, rather than the earlier LA-centered archive.
# =============================================================================

MODEL_CLASS = "2A_BF_cGANRandomVecFixed"
TARGET_KIND = "bf"
CONDITION_KIND = "lulc"

INSTALL_RASTERIO_IF_MISSING = True
MOUNT_GOOGLE_DRIVE = True

DRIVE_ARCHIVE_ZIP = "/content/drive/MyDrive/IM3/Full/Archive_CONUS_M1_random_stratified_2015.zip"
COPY_ARCHIVE_TO_LOCAL_BEFORE_EXTRACT = True
LOCAL_ARCHIVE_ZIP = "/content/Archive_CONUS_M1_random_stratified_2015.zip"
EXTRACT_ARCHIVE_IF_NEEDED = True
EXTRACT_DIR = "/home"

CONDITION_DIR = "/home/central"
TARGET_DIR = "/home/BFrac"
OUTPUT_ROOT = "/content/drive/MyDrive/IM3/EvalP1/revision_msa_sample"
OUTPUT_SUBDIR = "2A_BF_cGANRandomVecFixed_MSASample"

LULC_MODE = "raw"
NLCD_CLASSES = (
    11, 12, 21, 22, 23, 24, 31, 41,
    42, 43, 52, 71, 81, 82, 90, 95,
)

HEIGHT_UNITS = "feet"
MAX_HEIGHT_M = 75.0

LEARNING_RATES = [0.0001, 0.0002, 0.0005, 0.001]
EPOCHS = 1000
BATCH_SIZE = 128
NUM_WORKERS = 12
SAVE_EVERY = 250
PRINT_EVERY = 25
CACHE_DATA_IN_RAM = True
SEED = 2026
USE_AMP = True
LAMBDA_L1 = 100.0

GAN_LOSS = "bce"
BETA1 = 0.5
BETA2 = 0.999
LATENT_DIM = 8
INDEPENDENT_LEARNING_RATE_RUNS = True


In [4]:
rasterio = None


def ensure_rasterio() -> None:
    global rasterio
    try:
        import rasterio as _rasterio
    except ImportError:
        if not INSTALL_RASTERIO_IF_MISSING:
            raise
        print("Installing rasterio in this Colab runtime...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "rasterio"])
        import rasterio as _rasterio
    rasterio = _rasterio


def maybe_mount_drive() -> None:
    if not MOUNT_GOOGLE_DRIVE:
        return
    try:
        from google.colab import drive
    except ImportError:
        print("google.colab is unavailable; skipping Drive mount.")
        return
    drive.mount("/content/drive")


def resolve_drive_archive_path(configured_path: str) -> Path:
    configured = Path(configured_path)
    candidates = [configured]
    path_str = str(configured)
    if "My Drive" in path_str:
        candidates.append(Path(path_str.replace("My Drive", "MyDrive")))
    if "MyDrive" in path_str:
        candidates.append(Path(path_str.replace("MyDrive", "My Drive")))

    for candidate in candidates:
        if candidate.exists():
            return candidate

    basename = configured.name
    search_roots = [
        Path("/content/drive/MyDrive"),
        Path("/content/drive/My Drive"),
        Path("/content/drive/Shareddrives"),
        Path("/content/drive/Shared drives"),
    ]
    for root in search_roots:
        if not root.exists():
            continue
        matches = sorted(root.rglob(basename))
        if matches:
            print(f"Resolved archive by filename search: {matches[0]}")
            return matches[0]

    searched = "\n".join(str(path) for path in candidates)
    raise FileNotFoundError(
        "Archive zip not found. Checked the configured path variants:\n"
        f"{searched}\n"
        "Update DRIVE_ARCHIVE_ZIP to the exact mounted Google Drive path."
    )


def directory_has_tifs(path: Path) -> bool:
    return path.exists() and any(path.glob("*.tif*"))


def expected_data_ready() -> bool:
    return directory_has_tifs(Path(CONDITION_DIR)) and directory_has_tifs(Path(TARGET_DIR))


def maybe_extract_archive() -> None:
    if expected_data_ready():
        print(f"Using existing extracted folders: {CONDITION_DIR} and {TARGET_DIR}")
        return
    if not EXTRACT_ARCHIVE_IF_NEEDED:
        raise FileNotFoundError(
            f"Expected extracted data folders were not found: {CONDITION_DIR} and {TARGET_DIR}"
        )
    archive_path = resolve_drive_archive_path(DRIVE_ARCHIVE_ZIP)
    archive_to_extract = archive_path
    if COPY_ARCHIVE_TO_LOCAL_BEFORE_EXTRACT:
        local_archive_path = Path(LOCAL_ARCHIVE_ZIP)
        if not local_archive_path.exists() or local_archive_path.stat().st_size != archive_path.stat().st_size:
            local_archive_path.parent.mkdir(parents=True, exist_ok=True)
            print(f"Copying archive from Drive to local runtime: {local_archive_path}")
            shutil.copy2(archive_path, local_archive_path)
        archive_to_extract = local_archive_path
    extract_dir = Path(EXTRACT_DIR)
    extract_dir.mkdir(parents=True, exist_ok=True)
    print(f"Extracting {archive_to_extract} to {extract_dir}...")
    with zipfile.ZipFile(archive_to_extract) as zf:
        zf.extractall(extract_dir)
    print("Archive extraction complete.")
    if not expected_data_ready():
        raise FileNotFoundError(
            f"Archive extraction completed, but expected folders are still missing: {CONDITION_DIR} and {TARGET_DIR}"
        )


def set_seed(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


def worker_init_fn(worker_id: int) -> None:
    worker_seed = torch.initial_seed() % 2**32
    np.random.seed(worker_seed + worker_id)
    random.seed(worker_seed + worker_id)


def format_lr(value: float) -> str:
    return f"{value:g}".replace(".", "p")


def output_base_dir() -> Path:
    out = Path(OUTPUT_ROOT) / OUTPUT_SUBDIR
    out.mkdir(parents=True, exist_ok=True)
    return out


def output_run_dir(learning_rate: float, run_seed: int) -> Path:
    out = output_base_dir() / f"lr_{format_lr(learning_rate)}_seed_{run_seed}"
    out.mkdir(parents=True, exist_ok=True)
    return out


def list_tifs(folder: Path) -> set:
    return {path.name for path in folder.iterdir() if path.suffix.lower() in {".tif", ".tiff"}}


def read_raster(path: Path) -> np.ndarray:
    if rasterio is None:
        ensure_rasterio()
    with rasterio.open(path) as src:
        return src.read(1)


def should_use_amp(device: torch.device) -> bool:
    if not USE_AMP or device.type != "cuda":
        return False
    major, _minor = torch.cuda.get_device_capability()
    if major < 7:
        print("Disabling AMP for this GPU because compute capability is below 7.0.")
        return False
    return True


In [5]:
def encode_lulc_raw(arr: np.ndarray) -> np.ndarray:
    arr = arr.astype(np.float32)
    arr[(arr < 0) | (arr > 95)] = 0
    return ((arr / 95.0) * 2.0 - 1.0)[None, :, :].astype(np.float32)


def encode_lulc_onehot(arr: np.ndarray, classes: Iterable[int]) -> np.ndarray:
    arr = arr.astype(np.int16)
    channels = [(arr == cls).astype(np.float32) for cls in classes]
    return np.stack(channels, axis=0)


def encode_bf_unit(arr: np.ndarray) -> np.ndarray:
    arr = arr.astype(np.float32)
    arr = np.nan_to_num(arr, nan=0.0, posinf=1.0, neginf=0.0)
    return np.clip(arr, 0.0, 1.0)


def encode_bf_scaled(arr: np.ndarray) -> np.ndarray:
    return ((encode_bf_unit(arr) * 2.0) - 1.0)[None, :, :].astype(np.float32)


def height_to_meters(arr: np.ndarray, units: str) -> np.ndarray:
    arr = arr.astype(np.float32)
    if units == "feet":
        arr = arr * 0.3048
    elif units != "meters":
        raise ValueError("HEIGHT_UNITS must be 'feet' or 'meters'.")
    return arr


def encode_height_log1p(arr: np.ndarray, units: str, max_height_m: float) -> np.ndarray:
    arr = height_to_meters(arr, units)
    arr = np.nan_to_num(arr, nan=0.0, posinf=max_height_m, neginf=0.0)
    arr = np.clip(arr, 0.0, max_height_m)
    scaled = (np.log1p(arr) / np.log1p(max_height_m)) * 2.0 - 1.0
    return scaled[None, :, :].astype(np.float32)


def decode_height_log1p(scaled: np.ndarray, max_height_m: float) -> np.ndarray:
    clipped = np.clip(scaled, -1.0, 1.0)
    return np.expm1((clipped + 1.0) * np.log1p(max_height_m) / 2.0)


def encode_condition(arr: np.ndarray) -> np.ndarray:
    if CONDITION_KIND == "lulc":
        if LULC_MODE == "onehot":
            return encode_lulc_onehot(arr, NLCD_CLASSES)
        return encode_lulc_raw(arr)
    if CONDITION_KIND == "bf":
        return encode_bf_scaled(arr)
    raise ValueError(f"Unknown CONDITION_KIND: {CONDITION_KIND}")


def encode_target(arr: np.ndarray) -> np.ndarray:
    if TARGET_KIND == "bf":
        return encode_bf_scaled(arr)
    if TARGET_KIND == "height":
        return encode_height_log1p(arr, HEIGHT_UNITS, MAX_HEIGHT_M)
    raise ValueError(f"Unknown TARGET_KIND: {TARGET_KIND}")


class RasterPairDataset(Dataset):
    def __init__(self, condition_dir: Path, target_dir: Path, cache_data: bool = False) -> None:
        self.condition_dir = condition_dir
        self.target_dir = target_dir
        common = sorted(list_tifs(condition_dir) & list_tifs(target_dir))
        if not common:
            raise ValueError(f"No matching .tif files found in {condition_dir} and {target_dir}")
        self.files = common
        self.cache = None
        if cache_data:
            print("Caching all tiles in RAM...")
            self.cache = [self._load_item(filename) for filename in self.files]

    def __len__(self) -> int:
        return len(self.files)

    def _load_item(self, filename: str):
        cond = encode_condition(read_raster(self.condition_dir / filename))
        target = encode_target(read_raster(self.target_dir / filename))
        if cond.shape[-2:] != target.shape[-2:]:
            raise ValueError(f"Shape mismatch for {filename}: {cond.shape} vs {target.shape}")
        return torch.from_numpy(cond), torch.from_numpy(target)

    def __getitem__(self, idx: int):
        if self.cache is not None:
            return self.cache[idx]
        return self._load_item(self.files[idx])


In [6]:
class EncoderBlock(nn.Module):
    def __init__(self, in_channels: int, out_channels: int, norm: bool = True) -> None:
        super().__init__()
        layers = [nn.Conv2d(in_channels, out_channels, kernel_size=4, stride=2, padding=1, bias=not norm)]
        if norm:
            layers.append(nn.BatchNorm2d(out_channels))
        layers.append(nn.LeakyReLU(0.2, inplace=False))
        self.block = nn.Sequential(*layers)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.block(x)


class DecoderBlock(nn.Module):
    def __init__(self, in_channels: int, out_channels: int, dropout: bool = False) -> None:
        super().__init__()
        self.relu = nn.ReLU(inplace=False)
        self.deconv = nn.ConvTranspose2d(in_channels, out_channels, kernel_size=4, stride=2, padding=1, bias=False)
        self.bn = nn.BatchNorm2d(out_channels)
        self.dropout = nn.Dropout2d(0.5) if dropout else None

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = self.relu(x)
        x = self.deconv(x)
        x = self.bn(x)
        if self.dropout is not None:
            x = self.dropout(x)
        return x


class GeneratorRandomVec(nn.Module):
    def __init__(self, condition_channels: int, latent_dim: int = 8) -> None:
        super().__init__()
        self.latent_dim = latent_dim
        self.encoder1 = EncoderBlock(condition_channels + latent_dim, 64, norm=False)
        self.encoder2 = EncoderBlock(64, 128)
        self.encoder3 = EncoderBlock(128, 256)
        self.encoder4 = EncoderBlock(256, 512)
        self.encoder5 = EncoderBlock(512, 512)
        self.encoder6 = EncoderBlock(512, 512)
        self.encoder7 = EncoderBlock(512, 512)
        self.encoder8 = EncoderBlock(512, 512, norm=False)
        self.decoder8 = DecoderBlock(512, 512, dropout=True)
        self.decoder7 = DecoderBlock(1024, 512, dropout=True)
        self.decoder6 = DecoderBlock(1024, 512, dropout=True)
        self.decoder5 = DecoderBlock(1024, 512)
        self.decoder4 = DecoderBlock(1024, 256)
        self.decoder3 = DecoderBlock(512, 128)
        self.decoder2 = DecoderBlock(256, 64)
        self.decoder1 = nn.Sequential(nn.ReLU(inplace=False), nn.ConvTranspose2d(128, 1, kernel_size=4, stride=2, padding=1), nn.Tanh())

    def forward(self, cond: torch.Tensor, z: torch.Tensor) -> torch.Tensor:
        z_img = z[:, :, None, None].expand(-1, -1, cond.size(2), cond.size(3))
        x = torch.cat([cond, z_img], dim=1)
        e1 = self.encoder1(x)
        e2 = self.encoder2(e1)
        e3 = self.encoder3(e2)
        e4 = self.encoder4(e3)
        e5 = self.encoder5(e4)
        e6 = self.encoder6(e5)
        e7 = self.encoder7(e6)
        e8 = self.encoder8(e7)
        d8 = torch.cat([self.decoder8(e8), e7], dim=1)
        d7 = torch.cat([self.decoder7(d8), e6], dim=1)
        d6 = torch.cat([self.decoder6(d7), e5], dim=1)
        d5 = torch.cat([self.decoder5(d6), e4], dim=1)
        d4 = torch.cat([self.decoder4(d5), e3], dim=1)
        d3 = torch.cat([self.decoder3(d4), e2], dim=1)
        d2 = torch.cat([self.decoder2(d3), e1], dim=1)
        return self.decoder1(d2)


class DiscriminatorBlock(nn.Module):
    def __init__(self, in_channels: int, out_channels: int, norm: bool = True) -> None:
        super().__init__()
        layers = [nn.Conv2d(in_channels, out_channels, kernel_size=4, stride=2, padding=1, bias=not norm)]
        if norm:
            layers.append(nn.InstanceNorm2d(out_channels, affine=True))
        layers.append(nn.LeakyReLU(0.2, inplace=False))
        self.block = nn.Sequential(*layers)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.block(x)


class PatchDiscriminator(nn.Module):
    def __init__(self, condition_channels: int) -> None:
        super().__init__()
        self.block1 = DiscriminatorBlock(condition_channels + 1, 64, norm=False)
        self.block2 = DiscriminatorBlock(64, 128)
        self.block3 = DiscriminatorBlock(128, 256)
        self.block4 = DiscriminatorBlock(256, 512)
        self.final = nn.Conv2d(512, 1, kernel_size=4, stride=1, padding=1)

    def forward(self, target: torch.Tensor, cond: torch.Tensor) -> torch.Tensor:
        x = torch.cat([target, cond], dim=1)
        x = self.block1(x)
        x = self.block2(x)
        x = self.block3(x)
        x = self.block4(x)
        return self.final(x)


In [7]:
def adversarial_loss(pred: torch.Tensor, is_real: bool, loss_type: str) -> torch.Tensor:
    target = torch.ones_like(pred) if is_real else torch.zeros_like(pred)
    if loss_type == "bce":
        return F.binary_cross_entropy_with_logits(pred, target)
    if loss_type == "lsgan":
        return F.mse_loss(pred, target)
    raise ValueError(f"Unknown GAN_LOSS: {loss_type}")


In [8]:
@dataclass
class EpochMetrics:
    epoch: int
    lr: float
    d_loss: float
    g_loss: float
    adv_loss: float
    l1_loss: float
    seconds: float


def train_one_learning_rate(dataset: RasterPairDataset, condition_channels: int, learning_rate: float, run_seed: int) -> None:
    set_seed(run_seed)
    run_dir = output_run_dir(learning_rate, run_seed)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    use_amp = should_use_amp(device)
    generator = GeneratorRandomVec(condition_channels=condition_channels, latent_dim=LATENT_DIM).to(device)
    discriminator = PatchDiscriminator(condition_channels=condition_channels).to(device)
    g_optimizer = torch.optim.Adam(generator.parameters(), lr=learning_rate, betas=(BETA1, BETA2))
    d_optimizer = torch.optim.Adam(discriminator.parameters(), lr=learning_rate, betas=(BETA1, BETA2))
    loader_generator = torch.Generator()
    loader_generator.manual_seed(run_seed)
    dataloader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS, pin_memory=(device.type == "cuda"), persistent_workers=NUM_WORKERS > 0, worker_init_fn=worker_init_fn, generator=loader_generator)
    scaler = torch.amp.GradScaler("cuda", enabled=use_amp)
    metadata = {
        "model_class": MODEL_CLASS,
        "target_kind": TARGET_KIND,
        "condition_kind": CONDITION_KIND,
        "learning_rate": learning_rate,
        "seed": run_seed,
        "condition_channels": condition_channels,
        "latent_dim": LATENT_DIM,
        "independent_learning_rate_runs": True,
        "diversity_loss": False,
        "height_scaling": "log1p/expm1" if TARGET_KIND == "height" else None,
        "output_dir": str(run_dir),
    }
    with (run_dir / "metadata.json").open("w", encoding="utf-8") as f:
        json.dump(metadata, f, indent=2, sort_keys=True)
    metrics = []
    for epoch in range(1, EPOCHS + 1):
        generator.train(); discriminator.train()
        start = time.time()
        d_losses = []; g_losses = []; adv_losses = []; l1_losses = []
        for cond, real in dataloader:
            cond = cond.to(device=device, dtype=torch.float32, non_blocking=True)
            real = real.to(device=device, dtype=torch.float32, non_blocking=True)
            batch_size = cond.size(0)
            d_optimizer.zero_grad(set_to_none=True)
            with torch.no_grad():
                z_d = torch.randn(batch_size, LATENT_DIM, device=device)
                fake_d = generator(cond, z_d)
            with torch.amp.autocast("cuda", enabled=use_amp):
                real_pred = discriminator(real, cond)
                fake_pred = discriminator(fake_d.detach(), cond)
                d_loss = 0.5 * (adversarial_loss(real_pred, True, GAN_LOSS) + adversarial_loss(fake_pred, False, GAN_LOSS))
            scaler.scale(d_loss).backward(); scaler.step(d_optimizer)
            g_optimizer.zero_grad(set_to_none=True)
            z_g = torch.randn(batch_size, LATENT_DIM, device=device)
            with torch.amp.autocast("cuda", enabled=use_amp):
                fake = generator(cond, z_g)
                pred = discriminator(fake, cond)
                adv_loss = adversarial_loss(pred, True, GAN_LOSS)
                l1_loss = F.l1_loss(fake, real)
                g_loss = adv_loss + LAMBDA_L1 * l1_loss
            scaler.scale(g_loss).backward(); scaler.step(g_optimizer); scaler.update()
            d_losses.append(float(d_loss.detach().cpu()))
            g_losses.append(float(g_loss.detach().cpu()))
            adv_losses.append(float(adv_loss.detach().cpu()))
            l1_losses.append(float(l1_loss.detach().cpu()))
        row = EpochMetrics(epoch=epoch, lr=learning_rate, d_loss=float(np.mean(d_losses)), g_loss=float(np.mean(g_losses)), adv_loss=float(np.mean(adv_losses)), l1_loss=float(np.mean(l1_losses)), seconds=time.time() - start)
        metrics.append(row)
        if epoch % PRINT_EVERY == 0 or epoch == 1 or epoch == EPOCHS:
            print(f"[{MODEL_CLASS} lr={learning_rate:g}] epoch {epoch:04d}/{EPOCHS} D={row.d_loss:.4f} G={row.g_loss:.4f} adv={row.adv_loss:.4f} L1={row.l1_loss:.4f} sec={row.seconds:.1f}", flush=True)
        if epoch % SAVE_EVERY == 0 or epoch == EPOCHS:
            torch.save(generator.state_dict(), run_dir / f"generator_epoch_{epoch:04d}.pth")
            torch.save(discriminator.state_dict(), run_dir / f"discriminator_epoch_{epoch:04d}.pth")
    with (run_dir / "losses.csv").open("w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=list(asdict(metrics[0]).keys()))
        writer.writeheader()
        for row in metrics:
            writer.writerow(asdict(row))


def main() -> None:
    print(f"Using torch {torch.__version__}")
    if torch.cuda.is_available():
        print(torch.cuda.get_device_name(0)); torch.backends.cudnn.benchmark = True
    maybe_mount_drive(); maybe_extract_archive(); ensure_rasterio()
    condition_channels = len(NLCD_CLASSES) if CONDITION_KIND == "lulc" and LULC_MODE == "onehot" else 1
    dataset = RasterPairDataset(Path(CONDITION_DIR), Path(TARGET_DIR), cache_data=CACHE_DATA_IN_RAM)
    print(f"Found {len(dataset)} matched training tiles.")
    print(f"Output base: {output_base_dir()}")
    for idx, lr in enumerate(LEARNING_RATES):
        train_one_learning_rate(dataset, condition_channels, lr, SEED + idx * 1000)
    print("Training complete.")


In [ ]:
# Start this notebook's training/fitting job.
main()


Using torch 2.11.0+cu128
NVIDIA A100-SXM4-80GB
Mounted at /content/drive
Copying archive from Drive to local runtime: /content/Archive_CONUS_M1_random_stratified_2015.zip
Extracting /content/Archive_CONUS_M1_random_stratified_2015.zip to /home...
Archive extraction complete.
Caching all tiles in RAM...
Found 8000 matched training tiles.
Output base: /content/drive/MyDrive/IM3/EvalP1/revision_msa_sample/2A_BF_cGANRandomVecFixed_MSASample
[2A_BF_cGANRandomVecFixed lr=0.0001] epoch 0001/1000 D=0.3740 G=41.0390 adv=1.7120 L1=0.3933 sec=50.2
[2A_BF_cGANRandomVecFixed lr=0.0001] epoch 0025/1000 D=0.4570 G=14.6274 adv=1.6490 L1=0.1298 sec=10.4
[2A_BF_cGANRandomVecFixed lr=0.0001] epoch 0050/1000 D=0.5181 G=13.8879 adv=1.3955 L1=0.1249 sec=10.4
[2A_BF_cGANRandomVecFixed lr=0.0001] epoch 0075/1000 D=0.5221 G=12.5284 adv=1.2078 L1=0.1132 sec=10.4
[2A_BF_cGANRandomVecFixed lr=0.0001] epoch 0100/1000 D=0.5323 G=11.4067 adv=1.2439 L1=0.1016 sec=10.5
[2A_BF_cGANRandomVecFixed lr=0.0001] epoch 0125/1